# 16. Sources: downloads and checksums: exercises

Exercises of [chapter 16](../16_sources.md). Run the setup cell first. Everything here runs offline.

Work in a copy of this notebook: it is regenerated by `docs/manual/tools/build_notebooks.py`.

In [ ]:
# Setup: run this cell first, in every notebook
import os, subprocess, sys
from pathlib import Path

# find the repository: from this folder upwards, the first one holding data_acquisition/
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_acquisition").is_dir())
sys.path.insert(0, str(ROOT))                  # so that Python can import the pipeline
SAMPLE = ROOT / "output/manual/sample_root"    # the course's invented dataset
if not SAMPLE.exists():                        # generated once, with this same Python
    subprocess.run([sys.executable, str(ROOT / "docs/manual/tools/make_sample.py"), "--out", str(SAMPLE)], check=True)
DATASET = SAMPLE / "repo/doi_10.5555_training.pond"    # the sample's dataset folder
EXTRACT = ROOT / "docs/manual/data/montana_extract"    # a real piece of Rumney Ranch
os.environ["BEAVER_SITES"] = str(SAMPLE / "sites.yaml")    # use the sample's site registry, not the real one
import warnings; warnings.simplefilter("ignore")           # hide warnings; remove this line when debugging
print("repository:", ROOT); print("sample data root:", SAMPLE)

In [ ]:
import base64, csv, hashlib, http.client, http.server, inspect, io, json, tempfile, threading, zipfile
from data_acquisition import intake
from data_acquisition.sources import (archives, checksums, ckan, dataone, dataset_folder, downloads, hydroshare,
                                      pangaea, zenodo)
from data_acquisition.sources import http as web

SCRATCH = Path(tempfile.mkdtemp())
source = SCRATCH / "published" / "data.bin"            # the "repository's" copy of a file, served as file://
source.parent.mkdir()
source.write_bytes(os.urandom(12 * 1024 * 1024 + 123))
print(source, source.stat().st_size, "bytes")

# A fake repository on this machine, from exercise 6 on: ROUTES maps a path to the bytes it serves.
ROUTES, STATES_MD5 = {}, set()


class Repository(http.server.BaseHTTPRequestHandler):
    def respond(self, body=True):
        path = self.path.split("?")[0]
        if path not in ROUTES:
            return self.send_error(404)
        self.send_response(200)
        self.send_header("Content-Length", str(len(ROUTES[path])))
        if path in STATES_MD5:
            self.send_header("Content-MD5", base64.b64encode(hashlib.md5(ROUTES[path]).digest()).decode())
        self.end_headers()
        if body:
            self.wfile.write(ROUTES[path])

    def do_GET(self):
        self.respond()

    def do_HEAD(self):
        self.respond(body=False)

    def log_message(self, *args):
        pass


server = http.server.ThreadingHTTPServer(("127.0.0.1", 0), Repository)
threading.Thread(target=server.serve_forever, daemon=True).start()
BASE = f"http://127.0.0.1:{server.server_address[1]}"
DATA = b"some bytes\n" * 100
DATA_MD5 = hashlib.md5(DATA).hexdigest()
RESOURCE = "f" * 32
pack = io.BytesIO()
with zipfile.ZipFile(pack, "w") as z:
    z.writestr("inner/a.txt", "hello\n")
EML = (f'<eml:eml xmlns:eml="x" packageId="doi:10.5555/one"><dataset><title>From DataONE</title>'
       f'<otherEntity><entityName>ortho.txt</entityName><physical><size>{len(DATA)}</size>'
       f'<authentication method="MD5">{DATA_MD5}</authentication><distribution><online><url>{BASE}/f/ortho.txt</url>'
       f'</online></distribution></physical></otherEntity></dataset></eml:eml>')
TAB = ("/* DATA DESCRIPTION:\nCitation:\tRossi, A (2021): From PANGAEA [dataset]. PANGAEA, "
       "https://doi.org/10.1594/PANGAEA.9\nLicense:\tCC (CC-BY-4.0)\n*/\n"
       "Binary\tBinary (Hash)\tBinary (MIME type)\tBinary (Size)\n"
       f"ortho.txt\t{DATA_MD5}\ttext/plain\t1 kBytes\n")
REPOSITORY_RECORDS = {
    "/f/ortho.txt": DATA, "/f/data/contents/ortho.txt": DATA, "/pf/9/ortho.txt": DATA, "/c/pack.zip": pack.getvalue(),
    "/api/records/2": json.dumps({
        "doi": "10.5281/zenodo.2", "metadata": {"title": "From Zenodo", "creators": []},
        "files": [{"key": "ortho.txt", "size": len(DATA), "id": "1", "checksum": f"md5:{DATA_MD5}",
                   "links": {"self": f"{BASE}/f/ortho.txt"}}]}).encode(),
    "/d1/object/doi%3A10.5555%2Fone": EML.encode(),
    f"/hsapi2/resource/{RESOURCE}/json/": json.dumps({"title": "From HydroShare", "rights": {}}).encode(),
    f"/hsapi/resource/{RESOURCE}/files/": json.dumps({"next": None, "results": [
        {"url": f"{BASE}/f/data/contents/ortho.txt", "size": len(DATA), "checksum": DATA_MD5}]}).encode(),
    "/p/10.1594/PANGAEA.9": TAB.encode(),
    "/ck/package_show": json.dumps({"result": {"id": "pkg-1", "title": "From CKAN", "resources": [
        {"id": "r1", "format": "ZIP", "url": f"{BASE}/c/pack.zip", "size": str(len(pack.getvalue())),
         "hash": hashlib.md5(pack.getvalue()).hexdigest()}]}}).encode()}

**1. The contract in the model client.** Print `zenodo.py` and find, for each point of the contract, the line that does it.

In [ ]:
print(inspect.getsource(zenodo))

**2. Folders.** Name the folders of `10.5281/zenodo.1`, `https://doi.org/10.18739/A2CZ3253W`, `hs:abc123` and a CKAN package id.

In [ ]:
...  # your code here (dataset_folder(root, repository, identifier))

**3. Checksums.** Compute the file's MD5 and SHA-256; make its multipart ETag with 5 MiB parts and check it; check an ETag made with 7 MiB parts. Why is the answer None and not False?

In [ ]:
...  # your code here (checksums.digest, checksums._etag, checksums.check)

**4. Served MD5s.** Build response headers with `Content-MD5`, with `x-goog-hash`, and with a multipart `ETag`; what does `http.stated_md5` return for each?

In [ ]:
headers = http.client.HTTPMessage()
...  # your code here (base64 of the raw 16-byte digest)

**5. Download, resume, finish.** Leave a stale `.part` of the file in a scratch root, fetch it from its `file://` link, finish it against its MD5, log it, and read the log row. Then finish a copy against a wrong MD5: what happens to the file?

In [ ]:
root = SCRATCH / "root"
dest = dataset_folder(root, "example", "10.5555/example") / "data.bin"
...  # your code here (web.fetch, downloads.finish, downloads.log)

**6. The HTTP helpers.** The setup of this notebook runs a small fake repository on your own machine (`BASE`, `ROUTES`). Fetch a file with `web.get`, ask `web.served` for it with and without a stated MD5, and try `web.https`. Then call `web.retry` on a function that fails twice, and `web.get` on a page that does not exist.

In [ ]:
ROUTES["/data.txt"] = DATA
...  # your code here (set web.WAITS = (0, 0) first, or the retries wait for minutes)

**7. Verifying and logging.** Normalise a few checksums with `checksums.parse`. Then run `downloads.verify` with a record's MD5, with a multipart ETag it cannot decide (the server stating an MD5), with nothing and a silent server, and with `size_only`. Make a file read-only with `downloads.read_only`. Print a record with `downloads.listing`, with a filter and with checks. Finally, log the same path twice and read it with `downloads.read_log`.

In [ ]:
scratch = Path(tempfile.mkdtemp())
path = scratch / "data.txt"
path.write_bytes(DATA)
...  # your code here

**8. Archives.** Make a zip, extract it with `archives.extract`, and check a member with `archives.verify_member`. Extract again; then change an extracted file and do both once more. What if a file is not a member, or not a zip?

In [ ]:
scratch = Path(tempfile.mkdtemp())
...  # your code here (zipfile.ZipFile(path, "w").writestr(name, text))

**9. Five clients, one contract.** Point each client at the fake repository, and run its `record_of` and `download` into a scratch root (Zenodo, DataONE, HydroShare, PANGAEA, CKAN). What is the same in each, and what does each add? Try `ckan.package_id` and `hydroshare.list_files`.

In [ ]:
root = Path(tempfile.mkdtemp())
...  # your code here: REPOSITORY_RECORDS below the setup fills ROUTES; set zenodo.RECORD and so on

**10. Intake.** Write an intake prompt with `intake.write_prompt` for one link and for two records forming one dataset. Where does it tell Claude to stop and wait?

In [ ]:
out = Path(tempfile.mkdtemp())
...  # your code here